In [ ]:
# Install dependencies

!pip install -q transformers datasets peft accelerate bitsandbytes trl pyarrow evaluate

In [ ]:
# Imports and Seeds

import os
import re
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Load and split dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": "/kaggle/input/datasets/dhrvdng/multilingual-train/train_en_hi_ru.csv",
        "test":  "/kaggle/input/datasets/dhrvdng/multilingual-val/val_en_hi_ru.csv",
    }
)

train_dataset = dataset["train"]
test_dataset  = dataset["test"]

train_dataset = train_dataset.rename_column("label", "toxic")
test_dataset = test_dataset.rename_column("label", "toxic")

#dataset = dataset.shuffle(seed=SEED)
#splits  = dataset.train_test_split(test_size=0.1, seed=SEED)
#train_dataset = splits["train"]
#test_dataset  = splits["test"]

#train_dataset.to_parquet("/kaggle/input/datasets/dhrvdng/multilingual-train/train_en_hi_ru.csv")
#test_dataset.to_parquet("/kaggle/working/test_split.parquet")
 
#print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")
 
# Check class balance
toxic_count     = sum(train_dataset["toxic"])
non_toxic_count = len(train_dataset) - toxic_count
print(f"Toxic: {toxic_count}  |  Non-toxic: {non_toxic_count}")

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/dhrvdng/multilingual-val/val_en_hi_ru.csv")
print(df.columns.tolist())
print(df.head(2))

In [ ]:
# Load qwen tokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
 
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",   # right-pad for seq classification
)
 
# Qwen doesn't set a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token     = tokenizer.eos_token
    tokenizer.pad_token_id  = tokenizer.eos_token_id

In [ ]:
lengths = [len(tokenizer.encode(t)) for t in train_dataset["text"]]
print(f"Max: {max(lengths)}, Mean: {int(sum(lengths)/len(lengths))}, 95th pct: {sorted(lengths)[int(len(lengths)*0.95)]}")

In [ ]:
# Tokenize

SYSTEM = "You are a multilingual binary toxicity classifier understanding hindi, hinglish, english and russian."
MAX_LEN = 128
 
def tokenize(batch):
    combined = [
        f"{SYSTEM}\nClassify: {t}"
        for t in batch["text"]
    ]
    enc = tokenizer(
        combined,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,          # DataCollatorWithPadding handles this
    )
    enc["labels"] = [int(x) for x in batch["toxic"]]
    return enc
 
train_tok = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
test_tok  = test_dataset.map(tokenize,  batched=True, remove_columns=test_dataset.column_names)
 
print("Sample keys:", train_tok[0].keys())
print("Input len sample:", len(train_tok[0]["input_ids"]))

In [ ]:
# Load qwen as sequence classifier

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32,
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# Lora Config

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_proj", "v_proj"],
)
 
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Class weight loss

n_total   = len(train_dataset)
n_toxic   = sum(train_dataset["toxic"])
n_nontox  = n_total - n_toxic
 
# Weight inversely proportional to frequency
weights = torch.tensor(
    [n_total / (2 * n_nontox), n_total / (2 * n_toxic)],
    dtype=torch.float32
).to(model.device)
 
print(f"Class weights — non-toxic: {weights[0]:.3f}, toxic: {weights[1]:.3f}")
 
class WeightedTrainer(Trainer):
    """Trainer subclass that applies class weights to cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Metrics

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1    = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    f1_tox = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    return {"accuracy": acc, "f1_weighted": f1, "f1_toxic": f1_tox}

In [ ]:
# Training

training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen-toxic-multilingual",
    num_train_epochs=3,
    per_device_train_batch_size=16,          # seq cls uses less VRAM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,          # effective batch = 32
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                       # warm up first 10 % of steps
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch_fused",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=SEED,
)

In [ ]:
import gc

# Clear anything unused
gc.collect()
torch.cuda.empty_cache()

# Check VRAM after clearing
free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"Free VRAM: {free / 1e9:.2f} GB")

In [ ]:
# Training

data_collator = DataCollatorWithPadding(tokenizer)
 
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

import torch

torch.compile(model)
trainer.train()

In [ ]:
# Check actual step speed
import torch
print(torch.cuda.utilization())  # should be 80-95%, if low = bottleneck is CPU

In [ ]:
# Save model

SAVE_PATH = "/kaggle/working/qwen-toxic-multilingual"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved to", SAVE_PATH)

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("hugging face")

# Push to HuggingFace as backup
from huggingface_hub import login
login(token=secret_value)
model.push_to_hub("dhrv-dng/qwen-toxic-classifier-multiligual")
tokenizer.push_to_hub("dhrv-dng/qwen-toxic-classifier-multilingual")

print("Saved locally + pushed to HuggingFace Hub")